In [10]:
from coffea.dataset_tools import rucio_utils, dataset_query
from constructors.filesets import rucio_queries
from loadmodule import loadFunction
writeJson = loadFunction("constructors/utils/util_functions.py", "writeJson")
loadJson = loadFunction("constructors/utils/util_functions.py", "loadJson")

query_name = 'ZToMuMu'
rucioQuery = getattr(rucio_queries, query_name)
client = rucio_utils.get_rucio_client()

XCACHE_PREFIX = "root://xcache/"
REPLACE_PREFIX = "root://cmsdcadisk.fnal.gov//dcache/uscmsdisk"

fileset = {}
process_counters = {}   # contador por short_name

for dataset_query_str, info in rucioQuery.items():
    short_name = info["short_name"]     # "SingleTop"
    metadata = info["meta"].copy()      # {"isMC": True, "xsec": ...}

    # inicializar contador
    if short_name not in process_counters:
        process_counters[short_name] = 0

    # incrementar contador
    process_counters[short_name] += 1
    unique_key = f"{short_name}_{process_counters[short_name]}"

    # crear entrada
    fileset[unique_key] = {
        "files": {},
        "metadata": {
            **metadata,
            "short_name": short_name   # 🔹 identidad física real
        }
    }

    outlist = rucio_utils.query_dataset(dataset_query_str)

    for query in outlist:
        replicas = dataset_query.rucio_utils.get_dataset_files_replicas(
            query,
            client=client
        )

        for rep in range(len(replicas[0])):
            root = replicas[0][rep][0]

            # normalización xcache
            if REPLACE_PREFIX in root:
                root = root.replace(REPLACE_PREFIX, XCACHE_PREFIX)
            else:
                if root.startswith("root://"):
                    path = root.split("/", 3)[3]
                    root = f"{XCACHE_PREFIX}/{path}"

            fileset[unique_key]["files"][root] = "Events"
writeJson("constructors/filesets/ZToMuMu_fileset.json", fileset)
fileset = loadJson("constructors/filesets/ZToMuMu_fileset.json")

for key in fileset:
    nfiles = len(fileset[key]["files"])
    print(f"{key}: {nfiles} .root files")